In [2]:
# -- Create directories for references/aligments (RUN JUST ONCE)
from pathlib import Path

# -- Create directories for references/alignments
path = Path('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411')

(path / "genomes").mkdir(parents=True, exist_ok=True)
(path / "alignments").mkdir(parents=True, exist_ok=True)

In [3]:
# -- Download human reference genome (RUN ONLY ONCE)
!wget -P # -- Download human reference genome (RUN ONLY ONCE)
!wget -P {path/"genomes"} \
https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz \
https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

--2026-04-15 12:39:44--  https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 881964081 (841M) [application/x-gzip]
Saving to: 'genomes/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz'

Homo_sapiens.GRCh38 100%[===================>] 841.11M  3.80MB/s    in 93s     

2026-04-15 12:41:17 (9.06 MB/s) - 'genomes/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz' saved [881964081/881964081]



In [5]:
# -- Uncompressed human reference genome and rename (RUN ONLY ONCE)

!gunzip -f {path/"genomes"}/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

!mv genomes/Homo_sapiens.GRCh38.dna.primary_assembly.fa {path/"genomes"}/hg38.fa # move  to genomes directory


mv: rename genomes/Homo_sapiens.GRCh38.dna.primary_assembly.fa to /Users/leandro/Desktop/github/NGS-data/genomes/hg38.fa: No such file or directory


In [13]:
# -- Index genome with minimap2 (RUN ONLY ONCE)

!minimap2 -d {path/"genomes"/"hg38.mmi"} {path/"genomes"/"Homo_sapiens.GRCh38.dna.primary_assembly.fa"}

[M::mm_idx_gen::147.690*0.46] collected minimizers
[M::mm_idx_gen::234.108*0.49] sorted minimizers
[M::main::307.032*0.41] loaded/built the index for 194 target sequence(s)
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 194
[M::mm_idx_stat::322.563*0.40] distinct minimizers: 100159079 (38.75% are singletons); average occurrences: 5.545; average spacing: 5.581; total length: 3099750718
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -d /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/genomes/hg38.mmi /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/genomes/Homo_sapiens.GRCh38.dna.primary_assembly.fa
[M::main] Real time: 323.402 sec; CPU: 129.226 sec; Peak RSS: 1.165 GB


In [ ]:
# Reads pre-preprocessing (search PCR primer sequence in R1 reads)
!fqgrep "CTCCATATATGGGCTATGAACTAATGACCCC" E3-control_S1_L001_R1_001.fastq.gz > matched_reads_E3.fastq
# same for other R1 files (done in terminal)

In [3]:
# Define data location

reads_dir = path/'merged'
fastq_files = sorted(reads_dir.glob("*.merged.fastq.gz"))

fastq_files

[PosixPath('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/E3-control_S1_L001.merged.fastq.gz'),
 PosixPath('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/Hs-sorted_S2_L001.merged.fastq.gz'),
 PosixPath('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/Rr-sorted_S3_L001.merged.fastq.gz')]

In [4]:
# Minimap2 alignment 
threads = 8
index = path/"genomes"/"hg38.mmi"
alignments = path/"alignments"

for fq in fastq_files:
    sample_name = fq.name.replace(".merged.fastq.gz", "")
    out_bam = alignments / f"{sample_name}.hg38.bam"

    print(f"Aligning {sample_name} → hg38")

    !minimap2 -ax sr -t {threads} {index} {fq} \
    | samtools view -bS - \
    | samtools sort -@ {threads} -o {out_bam}

    !samtools index {out_bam}

Aligning E3-control_S1_L001 → hg38
[WARNING] Indexing parameters (-k, -w or -H) overridden by parameters used in the prebuilt index.
[M::main::7.924*0.68] loaded/built the index for 194 target sequence(s)
[M::mm_mapopt_update::7.925*0.68] mid_occ = 1000
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 194
[M::mm_idx_stat::17.534*0.40] distinct minimizers: 100159079 (38.75% are singletons); average occurrences: 5.545; average spacing: 5.581; total length: 3099750718
[M::worker_pipeline::565.847*0.40] mapped 278710 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -ax sr -t 8 /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/genomes/hg38.mmi /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/E3-control_S1_L001.merged.fastq.gz
[M::main] Real time: 566.390 sec; CPU: 226.697 sec; Peak RSS: 1.455 GB
[bam_sort_core] merging from 0 files and 8 in-memory blocks...


python(69055) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Aligning Hs-sorted_S2_L001 → hg38


python(69056) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[WARNING] Indexing parameters (-k, -w or -H) overridden by parameters used in the prebuilt index.
[M::main::10.885*0.50] loaded/built the index for 194 target sequence(s)
[M::mm_mapopt_update::10.886*0.50] mid_occ = 1000
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 194
[M::mm_idx_stat::26.148*0.29] distinct minimizers: 100159079 (38.75% are singletons); average occurrences: 5.545; average spacing: 5.581; total length: 3099750718
[M::worker_pipeline::260.046*0.38] mapped 182286 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -ax sr -t 8 /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/genomes/hg38.mmi /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/Hs-sorted_S2_L001.merged.fastq.gz
[M::main] Real time: 260.555 sec; CPU: 98.551 sec; Peak RSS: 1.463 GB
[bam_sort_core] merging from 0 files and 8 in-memory blocks...


python(70080) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Aligning Rr-sorted_S3_L001 → hg38


python(70081) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[WARNING] Indexing parameters (-k, -w or -H) overridden by parameters used in the prebuilt index.
[M::main::9.492*0.57] loaded/built the index for 194 target sequence(s)
[M::mm_mapopt_update::9.492*0.57] mid_occ = 1000
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 194
[M::mm_idx_stat::18.025*0.38] distinct minimizers: 100159079 (38.75% are singletons); average occurrences: 5.545; average spacing: 5.581; total length: 3099750718
[M::worker_pipeline::338.904*0.42] mapped 345691 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -ax sr -t 8 /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/genomes/hg38.mmi /Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/Rr-sorted_S3_L001.merged.fastq.gz
[M::main] Real time: 339.459 sec; CPU: 144.153 sec; Peak RSS: 1.381 GB
[bam_sort_core] merging from 0 files and 8 in-memory blocks...


python(71377) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [7]:
# Basic mapping statistics

for bam in alignments.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    !samtools flagstat {bam}


===== Hs-sorted_S2_L001.hg38.bam =====
182776 + 0 in total (QC-passed reads + QC-failed reads)
182286 + 0 primary
0 + 0 secondary
490 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
136021 + 0 mapped (74.42% : N/A)
135531 + 0 primary mapped (74.35% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


python(82018) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



===== E3-control_S1_L001.hg38.bam =====
279874 + 0 in total (QC-passed reads + QC-failed reads)
278710 + 0 primary
0 + 0 secondary
1164 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
248898 + 0 mapped (88.93% : N/A)
247734 + 0 primary mapped (88.89% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


python(82019) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



===== Rr-sorted_S3_L001.hg38.bam =====
347254 + 0 in total (QC-passed reads + QC-failed reads)
345691 + 0 primary
0 + 0 secondary
1563 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
249757 + 0 mapped (71.92% : N/A)
248194 + 0 primary mapped (71.80% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


python(82022) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [9]:
# Count uniquely mapped reads (MAPQ >0)

for bam in alignments.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    !samtools view -q 1 -c {bam}


===== Hs-sorted_S2_L001.hg38.bam =====
125906


python(82435) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



===== E3-control_S1_L001.hg38.bam =====
233944


python(82440) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



===== Rr-sorted_S3_L001.hg38.bam =====
231105


python(82441) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [15]:
# Number of distinct genomic loci hit

import subprocess

for bam in alignments.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    cmd = f"samtools view {bam} | awk '{{print $3\":\"$4}}' | sort | uniq | wc -l"
    subprocess.run(cmd, shell=True)


===== Hs-sorted_S2_L001.hg38.bam =====


python(86516) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



===== E3-control_S1_L001.hg38.bam =====
  121663


python(86527) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  236845

===== Rr-sorted_S3_L001.hg38.bam =====


python(86539) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  211571


In [16]:
# Reads per chromosome

for bam in alignments.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    !samtools view {bam} | cut -f3 | sort | uniq -c


===== Hs-sorted_S2_L001.hg38.bam =====


python(86876) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


46755 *
14215 1
7062 10
6125 11
5245 12
3514 13
3852 14
3900 15
5045 16
6076 17
2518 18
4698 19
8971 2
3429 20
2248 21
3642 22
7855 3
5781 4
6596 5
6896 6
6878 7
5334 8
5307 9
  21 GL000008.2
  20 GL000009.2
  23 GL000194.1
   6 GL000195.1
  28 GL000205.2
   2 GL000208.1
   1 GL000213.1
 240 GL000214.1
 591 GL000216.2
   9 GL000218.1
   3 GL000219.1
  60 GL000220.1
  12 GL000221.1
  58 GL000224.1
1094 GL000225.1
   1 KI270320.1
   4 KI270330.1
   1 KI270333.1
  10 KI270438.1
  18 KI270442.1
   7 KI270467.1
   1 KI270508.1
   6 KI270519.1
   4 KI270521.1
   9 KI270706.1
   5 KI270707.1
  10 KI270708.1
  18 KI270709.1
  10 KI270711.1
  14 KI270712.1
   9 KI270713.1
   6 KI270714.1
   4 KI270719.1
   2 KI270720.1
  11 KI270721.1
   7 KI270722.1
   9 KI270723.1
  10 KI270724.1
   7 KI270725.1
   5 KI270727.1
  37 KI270728.1
  20 KI270729.1
   3 KI270730.1
  14 KI270731.1
   3 KI270732.1
  52 KI270733.1
  13 KI270734.1
  13 KI270736.1
   2 KI270737.1
   3 KI270738.1
  13 KI270742.1
  24 KI2

python(86882) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


30976 *
25394 1
12692 10
10957 11
9969 12
7677 13
7029 14
6712 15
8737 16
10109 17
5174 18
7231 19
17341 2
5840 20
4400 21
5735 22
15694 3
11749 4
12951 5
13783 6
13054 7
9680 8
9487 9
  35 GL000008.2
  40 GL000009.2
  48 GL000194.1
  24 GL000195.1
  47 GL000205.2
   5 GL000208.1
   4 GL000213.1
 184 GL000214.1
 439 GL000216.2
  19 GL000218.1
   9 GL000219.1
 128 GL000220.1
  27 GL000221.1
  74 GL000224.1
 792 GL000225.1
   5 GL000226.1
   1 KI270302.1
   1 KI270304.1
   1 KI270320.1
   4 KI270333.1
   2 KI270337.1
   1 KI270338.1
   1 KI270363.1
   2 KI270366.1
   1 KI270374.1
   1 KI270382.1
   1 KI270396.1
   1 KI270411.1
   6 KI270435.1
 171 KI270438.1
  76 KI270442.1
   1 KI270448.1
   3 KI270466.1
  10 KI270467.1
   1 KI270508.1
   1 KI270509.1
   3 KI270512.1
   1 KI270515.1
   5 KI270519.1
  18 KI270538.1
   1 KI270579.1
   1 KI270581.1
   1 KI270582.1
   9 KI270587.1
   5 KI270589.1
   3 KI270590.1
   1 KI270591.1
   1 KI270593.1
  28 KI270706.1
   8 KI270707.1
  12 KI270708.1

python(86892) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


97497 *
26315 1
13153 10
11502 11
9598 12
5874 13
7222 14
7217 15
9407 16
11907 17
4583 18
9235 19
16753 2
6387 20
4232 21
7031 22
14424 3
9943 4
12061 5
12539 6
12404 7
9495 8
9785 9
  26 GL000008.2
  27 GL000009.2
  42 GL000194.1
  12 GL000195.1
  54 GL000205.2
   3 GL000208.1
 397 GL000214.1
 926 GL000216.2
  20 GL000218.1
  16 GL000219.1
 110 GL000220.1
  19 GL000221.1
 102 GL000224.1
1782 GL000225.1
   2 KI270303.1
   4 KI270330.1
   2 KI270337.1
   1 KI270372.1
   2 KI270435.1
  12 KI270438.1
  24 KI270442.1
   1 KI270466.1
   9 KI270467.1
   6 KI270508.1
   3 KI270512.1
   5 KI270519.1
   3 KI270521.1
   1 KI270538.1
   1 KI270539.1
   3 KI270579.1
   1 KI270581.1
   1 KI270588.1
  28 KI270706.1
   7 KI270707.1
  11 KI270708.1
  38 KI270709.1
  14 KI270711.1
  30 KI270712.1
  19 KI270713.1
   8 KI270714.1
   1 KI270715.1
   1 KI270716.1
   1 KI270717.1
  16 KI270719.1
   2 KI270720.1
  26 KI270721.1
  16 KI270722.1
  17 KI270723.1
  16 KI270724.1
  16 KI270725.1
   3 KI270726.1


In [18]:
# Summary table

import pandas as pd
import subprocess

results = []

for bam in alignments.glob("*.bam"):
    total = int(subprocess.check_output(
        f"samtools view -c {bam}", shell=True
    ).decode().strip())

    unique = int(subprocess.check_output(
        f"samtools view -q 30 -c {bam}", shell=True
    ).decode().strip())

    multi = int(subprocess.check_output(
        f"samtools view -c -f 256 {bam}", shell=True
    ).decode().strip())

    results.append({
        "sample": bam.name,
        "total_reads": total,
        "unique_MAPQ30": unique,
        "secondary_alignments": multi
    })

df = pd.DataFrame(results)
df

python(87783) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87784) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87785) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87786) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87787) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87788) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87795) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87807) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87809) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


,sample,total_reads,unique_MAPQ30,secondary_alignments
0,Hs-sorted_S2_L001.hg38.bam,182776,93525,0
1,E3-control_S1_L001.hg38.bam,279874,205321,0
2,Rr-sorted_S3_L001.hg38.bam,347254,166134,0
